# Check data type in table :

In [0]:
df=spark.sql("select * from stockmarket.raw.timesofindia")
display(df)

# Text convert into time stamp (Pubdate)

In [0]:
from pyspark.sql.functions import to_timestamp, coalesce, regexp_replace, trim

df_clean = df.withColumn(
    "pubDate",
    coalesce(
        to_timestamp("pubDate", "yyyy-MM-dd'T'HH:mm:ssxxx"),       # 2026-04-28T17:28:04+05:30
        to_timestamp("pubDate", "yyyy-MM-dd'T'HH:mm:ssxx"),         # 2026-04-28T17:28:04+0530
        to_timestamp(
            trim(regexp_replace("pubDate", "^[A-Za-z]+,\\s*", "")),
            "dd MMM yyyy HH:mm:ss xx"                                # 28 Apr 2026 10:30:00 +0000
        ),
    )
)
display(df_clean)

# Remove duplites in table

In [0]:
cleandata = df_clean.dropDuplicates()
display(cleandata)

# Add data in table : stockmarket.sliver.timesofindia

In [0]:
cleandata.write.format("delta").mode("append").saveAsTable("stockmarket.sliver.timesofindia")